# DeepGuard AI — Training & Evaluation Notebook
## Deep Learning-Based Network Intrusion Detection and Threat Classification System
**Author:** Vivek (B.Tech Capstone Project)  
**Dataset:** NSL-KDD (Canadian Institute for Cybersecurity, UNB)  
**Model Architecture:** PyTorch Multilayer Perceptron (MLP)  
**Objective:** Build, train, evaluate, and export an end-to-end intrusion detection system.

---
### Notebook Pipeline Overview
1. **Phase 0:** Environment & Runtime Inspection
2. **Phase 1:** Dataset Acquisition & Verification
3. **Phase 2:** Exploratory Data Analysis & Preprocessing Pipeline
4. **Phase 3:** PyTorch MLP Model Training & Baseline Classifier
5. **Phase 4:** Comprehensive Performance Evaluation
6. **Phase 5:** Artifact Serialization & Download Package

## 1. Environment & Hardware Inspection
Verify the Python runtime, host operating system, and hardware acceleration (NVIDIA GPU or CPU).

In [ ]:
import sys
import platform
import subprocess
import torch

print(f"Python Version: {sys.version}")
print(f"Operating System: {platform.system()} {platform.release()}")
print(f"PyTorch Version: {torch.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Execution Device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Allocated Memory: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
else:
    print("Running on standard CPU (fully supported)")

## 2. Dependency Installation
Ensure all essential scientific computing, ML, and visualization packages are installed.

In [ ]:
# Install dependencies if running on Google Colab
!pip install -q torch torchvision scikit-learn pandas numpy joblib matplotlib

## 3. Dataset Acquisition (NSL-KDD)
Download the benchmark NSL-KDD training and test traffic sets directly from verified public repositories.

In [ ]:
import os
import urllib.request

os.makedirs("data", exist_ok=True)
os.makedirs("model", exist_ok=True)

DATASET_URLS = {
    "KDDTrain+.txt": "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain+.txt",
    "KDDTest+.txt": "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest+.txt",
}

for fname, url in DATASET_URLS.items():
    target_path = os.path.join("data", fname)
    if not os.path.exists(target_path):
        print(f"Downloading {fname} from {url}...")
        urllib.request.urlretrieve(url, target_path)
        print(f"Downloaded {fname} ({os.path.getsize(target_path):,} bytes)")
    else:
        print(f"{fname} already present locally ({os.path.getsize(target_path):,} bytes)")

## 4. Phase 2: Dataset Loading & Comprehensive Inspection
Define canonical column schemas and inspect data shapes, types, missing values, duplicates, and class distributions.

In [ ]:
import pandas as pd
import numpy as np

NSL_KDD_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label", "difficulty_level"
]

train_df = pd.read_csv("data/KDDTrain+.txt", header=None, names=NSL_KDD_COLUMNS, sep=",")
test_df = pd.read_csv("data/KDDTest+.txt", header=None, names=NSL_KDD_COLUMNS, sep=",")

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print(f"Missing values in train: {train_df.isnull().sum().sum()}")
print(f"Duplicate rows in train: {train_df.duplicated().sum()}")
print("\nFirst 5 rows of KDDTrain+:")
display(train_df.head())

print("\nTop 10 attack classes in raw training data:")
print(train_df["label"].value_counts().head(10))

## 5. Phase 2: Feature Selection & Target Leakage Removal

### Dropped Columns Rationale
- **`difficulty_level` (Column 42):** Assigned by UNB researchers in 2009 indicating how many ML algorithms correctly classified each instance. It is an artifact of the benchmark study, NOT part of network packet traffic. Retaining it would constitute severe target leakage.
- No IP addresses, timestamps, or session IDs are present in NSL-KDD.

In [ ]:
# Binary label mapping: normal -> 0 (Benign), everything else -> 1 (Attack)
y_raw_train = train_df["label"].astype(str).str.strip().str.lower().str.rstrip(".")
y_train_binary = (y_raw_train != "normal").astype(int).values

print(f"Binary Label Distribution (Train):")
print(f"  Benign (0): {(y_train_binary == 0).sum():,} ({(y_train_binary == 0).mean()*100:.2f}%)")
print(f"  Attack (1): {(y_train_binary == 1).sum():,} ({(y_train_binary == 1).mean()*100:.2f}%)")

# Drop difficulty_level and label from features
X_train_raw = train_df.drop(columns=["difficulty_level", "label"])
print(f"Remaining feature count: {X_train_raw.shape[1]} (Expected: 41)")

## 6. Phase 2: Train / Validation / Test Stratified Split
Split the data into 70% Train, 15% Validation, and 15% Test with stratification to preserve benign/attack balance across all splits.

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified 70/15/15 split
X_temp, X_test, y_temp, y_test = train_test_split(
    X_train_raw, y_train_binary, test_size=0.15, random_state=42, stratify=y_train_binary
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=(0.15 / 0.85), random_state=42, stratify=y_temp
)

print(f"Train set size:      {len(X_train):,} ({len(X_train)/len(X_train_raw)*100:.1f}%) | Attack ratio: {y_train.mean():.4f}")
print(f"Validation set size: {len(X_val):,} ({len(X_val)/len(X_train_raw)*100:.1f}%) | Attack ratio: {y_val.mean():.4f}")
print(f"Test set size:       {len(X_test):,} ({len(X_test)/len(X_train_raw)*100:.1f}%) | Attack ratio: {y_test.mean():.4f}")

## 7. Phase 2: Categorical Encoding & Feature Scaling (StandardScaler)

**Critical Rule:** The `StandardScaler` and categorical encoding dictionaries are fitted **ONLY** on the training split `X_train`. They are then applied to `X_val` and `X_test` without refitting.

In [ ]:
import json
import joblib
from sklearn.preprocessing import StandardScaler

CATEGORICAL_COLS = ["protocol_type", "service", "flag"]
NUMERICAL_COLS = [col for col in X_train.columns if col not in CATEGORICAL_COLS]

# 1. Compute numerical medians on training data only
numerical_medians = {col: float(X_train[col].median()) for col in NUMERICAL_COLS}

# 2. Compute categorical modes and mappings on training data only
categorical_modes = {}
category_mappings = {}
for col in CATEGORICAL_COLS:
    clean_series = X_train[col].dropna().astype(str).str.strip().str.lower()
    categorical_modes[col] = clean_series.mode().iloc[0] if not clean_series.empty else "unknown"
    unique_cats = sorted(clean_series.unique().tolist())
    mapping = {cat: idx for idx, cat in enumerate(unique_cats)}
    mapping["__unknown__"] = len(unique_cats)
    category_mappings[col] = mapping

def encode_features(df_subset):
    """Encode categoricals and impute missing values using train statistics."""
    res = pd.DataFrame(index=df_subset.index)
    for col in df_subset.columns:
        if col in CATEGORICAL_COLS:
            mapping = category_mappings[col]
            mode_val = categorical_modes[col]
            series = df_subset[col].fillna(mode_val).astype(str).str.strip().str.lower()
            res[col] = series.map(lambda v: mapping.get(v, mapping["__unknown__"])).astype(float)
        else:
            med = numerical_medians[col]
            res[col] = pd.to_numeric(df_subset[col], errors="coerce").fillna(med).astype(float)
    return res.to_numpy(dtype=np.float32)

# Encode splits
X_train_num = encode_features(X_train)
X_val_num = encode_features(X_val)
X_test_num = encode_features(X_test)

# Fit StandardScaler on training features only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)

# Apply fitted scaler to val and test
X_val_scaled = scaler.transform(X_val_num)
X_test_scaled = scaler.transform(X_test_num)

print(f"X_train_scaled: {X_train_scaled.shape} | mean={X_train_scaled.mean():.4f}, std={X_train_scaled.std():.4f}")
print(f"X_val_scaled:   {X_val_scaled.shape}")
print(f"X_test_scaled:  {X_test_scaled.shape}")
print(f"NaN check: Train={np.isnan(X_train_scaled).any()}, Val={np.isnan(X_val_scaled).any()}, Test={np.isnan(X_test_scaled).any()}")

## 8. Phase 2: Save Preprocessing Artifacts
Save the fitted `preprocessor.joblib`, `feature_schema.json`, and `label_mapping.json` for reproducible production inference.

In [ ]:
# Save StandardScaler
joblib.dump(scaler, "model/preprocessor.joblib")

# Save Feature Schema
schema = {
    "feature_names": list(X_train.columns),
    "feature_count": len(X_train.columns),
    "categorical_columns": CATEGORICAL_COLS,
    "numerical_columns": NUMERICAL_COLS,
    "target_column": "label",
    "drop_columns": ["difficulty_level"],
    "category_mappings": category_mappings,
    "categorical_modes": categorical_modes,
    "numerical_medians": numerical_medians,
}
with open("model/feature_schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)

# Save Label Mapping
label_mapping = {
    "normal": 0,
    "attack": 1,
    "mapping_description": "0 = Benign/Normal, 1 = Attack/Malicious"
}
with open("model/label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2)

print("Artifacts saved successfully:")
for f in ["model/preprocessor.joblib", "model/feature_schema.json", "model/label_mapping.json"]:
    print(f"  [OK] {f} ({os.path.getsize(f):,} bytes)")

## 9. Phase 2 Verification: Fresh-Runtime Load Test
Load the saved artifacts in fresh variables and verify shape and numeric integrity.

In [ ]:
# Verification cell
loaded_scaler = joblib.load("model/preprocessor.joblib")
with open("model/feature_schema.json", "r", encoding="utf-8") as f:
    loaded_schema = json.load(f)

sample_row = X_test_num[0:1]
transformed_sample = loaded_scaler.transform(sample_row)

assert loaded_scaler.n_features_in_ == loaded_schema["feature_count"], "Feature count mismatch!"
assert transformed_sample.shape == (1, loaded_schema["feature_count"]), "Shape mismatch!"
assert not np.isnan(transformed_sample).any(), "NaNs detected in sample!"
assert not np.isinf(transformed_sample).any(), "Infs detected in sample!"

print(f"ALL VERIFICATIONS PASSED!")
print(f"  Fitted features: {loaded_scaler.n_features_in_}")
print(f"  Sample transformed shape: {transformed_sample.shape}")
print(f"  First 5 scaled values: {transformed_sample[0, :5]}")